In [4]:
import torch
import torch.nn as nn

class CBOWEmbeddingModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim = 100, padding_idx = 0):
        super().__init__()
        
        self.embedd = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx,
        )
    
    def forward(self, context_ids):
        embedding = self.embedd(context_ids)
        return embedding

model = CBOWEmbeddingModel(vocab_size=3, embedding_dim=5)

context_ids = torch.tensor([
    [1, 2, 0],
    [2, 1, 2],
], dtype=torch.long)

embedding = model(context_ids)

print("词向量表形状：", model.embedd.weight.shape)
print("输入形状：", context_ids.shape)
print("输出形状：", embedding.shape)
print("查表结果：\n", embedding)


model = CBOWEmbeddingModel(
    vocab_size=3,
    embedding_dim=5,
    padding_idx=0,
)

# 4 条样本，每条包含 4 个上下文位置
context_ids = torch.tensor([
    [1, 2, 1, 2],  # 没有 PAD
    [1, 2, 0, 0],  # 后两个位置是 PAD
    [2, 0, 0, 0],  # 只有一个真实词
    [0, 0, 0, 0],  # 全部是 PAD
], dtype=torch.long)

embedding = model(context_ids)

print("输入：")
print(context_ids)

print("\n输入形状：")
print(context_ids.shape)

print("\n输出形状：")
print(embedding.shape)

print("\n查表结果：")
print(embedding)

词向量表形状： torch.Size([3, 5])
输入形状： torch.Size([2, 3])
输出形状： torch.Size([2, 3, 5])
查表结果：
 tensor([[[ 0.1766,  0.0833,  1.1296, -0.5061, -0.4940],
         [ 0.1517, -0.9819,  1.3745,  0.7513,  0.8102],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000]],

        [[ 0.1517, -0.9819,  1.3745,  0.7513,  0.8102],
         [ 0.1766,  0.0833,  1.1296, -0.5061, -0.4940],
         [ 0.1517, -0.9819,  1.3745,  0.7513,  0.8102]]],
       grad_fn=<EmbeddingBackward0>)
输入：
tensor([[1, 2, 1, 2],
        [1, 2, 0, 0],
        [2, 0, 0, 0],
        [0, 0, 0, 0]])

输入形状：
torch.Size([4, 4])

输出形状：
torch.Size([4, 4, 5])

查表结果：
tensor([[[ 0.0541, -0.1345,  0.5216,  0.4566,  0.6564],
         [ 0.9312,  1.3931,  0.4528,  1.2689, -0.5792],
         [ 0.0541, -0.1345,  0.5216,  0.4566,  0.6564],
         [ 0.9312,  1.3931,  0.4528,  1.2689, -0.5792]],

        [[ 0.0541, -0.1345,  0.5216,  0.4566,  0.6564],
         [ 0.9312,  1.3931,  0.4528,  1.2689, -0.5792],
         [ 0.0000,  0.0000,  0.0000,  0.000

In [5]:
# ============================================================
# 用 gensim 训练自己的 Word2Vec 词向量
# 语料：sentence.txt（20 句中文），gensim 需要先分词
# ============================================================

import jieba
from gensim.models import Word2Vec

# 1. 读取语料（每行一句话）
with open("sentence.txt", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]
print(f"句子数量：{len(lines)}")

# 2. 用 jieba 分词 → gensim 要求「列表的列表」：[[词, 词, ...], [词, ...], ...]
sentences = [list(jieba.cut(line)) for line in lines]
print("分词示例：", sentences[0])

# 3. 训练 Word2Vec
model = Word2Vec(
    sentences=sentences,
    vector_size=32,     # 词向量维度（与 word2vec.txt 一致，方便对比）
    window=2,           # 上下文窗口大小
    min_count=1,        # 词频低于该值的词丢弃，这里保留全部
    sg=0,               # sg=0: CBOW  |  sg=1: Skip-gram
    epochs=100,         # 训练轮数
    seed=42,            # 固定随机种子，结果可复现
)

# 4. 查看训练结果
print("\n词表大小：", len(model.wv))

w = "模型"
print(f"\n「{w}」的词向量（前 8 维）：", model.wv[w][:8])

print(f"\n与「{w}」最相似的词：")
for word, sim in model.wv.most_similar(w, topn=5):
    print(f"  {word:<8} {sim:.4f}")

print("\n「模型」与「训练」的余弦相似度：", round(model.wv.similarity("模型", "训练"), 4))

# 5. 保存模型
model.save("gensim_cbow.model")                                   # 完整模型，可继续训练
model.wv.save_word2vec_format("word2vec_gensim.txt", binary=False)  # word2vec 纯文本格式
print("\n已保存：gensim_cbow.model / word2vec_gensim.txt")

# 6. 用 KeyedVectors 重新加载文本格式词向量（不依赖训练模型即可查询）
from gensim.models import KeyedVectors

kv = KeyedVectors.load_word2vec_format("word2vec_gensim.txt", binary=False)
print("\n重新加载的词表大小：", len(kv))
print("kv['模型'][:8] =", kv["模型"][:8])

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\13744\AppData\Local\Temp\jieba.cache


句子数量：20


Loading model cost 0.693 seconds.
Prefix dict has been built successfully.


分词示例： ['自然语言', '处理', '是', '人工智能', '的', '重要', '方向']

词表大小： 114

「模型」的词向量（前 8 维）： [-0.01509061 -0.00539538  0.06606323  0.00166975 -0.0022774   0.00699448
  0.01894512 -0.03133481]

与「模型」最相似的词：
  的        0.7215
  通常       0.7148
  向量       0.6645
  可以       0.6609
  文本       0.6547

「模型」与「训练」的余弦相似度： 0.4889

已保存：gensim_cbow.model / word2vec_gensim.txt

重新加载的词表大小： 114
kv['模型'][:8] = [-0.01509061 -0.00539538  0.06606323  0.00166975 -0.0022774   0.00699448
  0.01894512 -0.03133481]


# RNN 数学原理推导

> 目标：从递推公式出发，推完前向 → 损失 → BPTT 梯度 → 梯度消失/爆炸的根源。
> 视角：把 $t$ 看作时间采样点，隐状态 $h_t$ 就像滤波器的**状态变量**，整条链是一个带反馈的离散系统。

## 1. 记号与核心递推

输入序列 $\{x_1, x_2, \dots, x_T\}$，$x_t \in \mathbb{R}^{n}$；隐状态 $h_t \in \mathbb{R}^{d}$；输出 $y_t \in \mathbb{R}^{m}$（先忽略 batch 维）。

所有时刻**共享**一组参数：

- $W_{xh} \in \mathbb{R}^{d \times n}$，$W_{hh} \in \mathbb{R}^{d \times d}$，$b_h \in \mathbb{R}^{d}$
- $W_{hy} \in \mathbb{R}^{m \times d}$，$b_y \in \mathbb{R}^{m}$

**核心递推（RNN 之所以是 RNN）**：

$$
h_t = \tanh\!\left(W_{xh} x_t + W_{hh} h_{t-1} + b_h\right), \qquad h_0 = \mathbf{0}
$$

$$
\hat{y}_t = \mathrm{softmax}\!\left(W_{hy} h_t + b_y\right)
$$

关键点：$h_t$ 由「当前输入 $x_t$」和「上一时刻状态 $h_{t-1}$」共同决定 —— 这是一个**反馈回路**，权重在时间上复用（权重共享），所以能处理任意长度的序列。

## 2. 时间展开（Unfold）

把递推按时间轴展开：

```
h₀ → h₁ → h₂ → … → hₜ → … → h_T
      ↑     ↑           ↑
     x₁    x₂          x_T
```

$$
h_t = f(x_t,\, f(x_{t-1},\, f(\dots f(x_1, h_0)\dots)))
$$

也就是说 $h_t$ 是整个历史 $\{x_1..x_t\}$ 的**压缩表示**。

> 对照前面的 CBOW：CBOW 只用固定窗口内的一次性平均（无记忆）；RNN 是**因果累积**，$h_t$ 依赖全部历史。

### 2.1 单向 RNN 是因果系统

展开式说明了一个更强的性质：$h_t$ 的依赖范围**只到 $x_t$**，绝不涉及 $x_{t+1}$ 及以后。一般因果系统的严格定义是：若两个输入序列在所有 $k\le t$ 的位置完全相同，则它们在时刻 $t$ 的输出也相同。对于线性时不变（LTI）系统，这才等价于单位冲激响应 $g[n]=0,\ n<0$。RNN 的递推边只从 $h_{t-1}$ 指向 $h_t$，因此前向依赖是因果的。

**因果性带来的能力：**

- **流式/在线推理**：每来一个 $x_t$ 就更新 $h_t$、立即输出 $\hat{y}_t$，不用等整句说完 → 实时语音识别、流式翻译、TTS
- **自回归生成**：逐词生成时，每个 $\hat{y}_t$ 只用已生成的历史
- **前向依赖一致**：训练和推理时都不读取未来输入；但自回归训练若使用 teacher forcing，仍可能产生 exposure bias

**代价与对照：**

| 结构 | 因果性 | 典型用途 |
|------|:------:|----------|
| 单向 RNN | ✅ 因果 | 流式任务、文本生成 |
| 双向 RNN（ELMo） | ❌ 看全序列 | 完形填空式理解任务 |
| Transformer 自注意力 | ❌ 默认非因果 | 需 causal mask（下三角）恢复因果 |
| BERT | ❌ 双向 | 理解类任务 |

- 数学表述：依赖结构是**下三角**的 —— 第 $t$ 个输出只连到 $\le t$ 的输入
- 双向 RNN = 正向、反向各跑一遍再拼接，能看全上下文，但**不能流式使用**

> ⚠️ 易混点：BPTT 中 $\delta_t$ 是"从未来往过去"递推的，看起来反因果 —— 那只是**离线训练**时梯度计算的方向；**前向推理永远是因果的**。训练（离线看整句）与部署（在线只看历史）如果不一致，会带来 train/test 偏差。

### 2.2 双向 RNN 的数学形式

双向 RNN 使用两套独立参数，分别从左到右、从右到左递推：

$$
\overrightarrow{h_t}=\tanh(W_f x_t+U_f\overrightarrow{h_{t-1}}+b_f)
$$

$$
\overleftarrow{h_t}=\tanh(W_b x_t+U_b\overleftarrow{h_{t+1}}+b_b)
$$

同一位置的最终表示通常按特征维拼接：

$$
h_t^{\mathrm{bi}}=[\overrightarrow{h_t};\overleftarrow{h_t}]\in\mathbb{R}^{2d}
$$

正向分量只看当前位置及左侧，反向分量只看当前位置及右侧；拼接结果同时利用两边信息，所以整个双向 RNN 对原时间方向是**非因果的**，不适合无等待的流式预测。

## 3. 损失函数

以语言模型为例：用 $t$ 时刻的状态预测下一个词，每个时刻都有监督信号：

$$
L = \sum_{t=1}^{T} L_t, \qquad L_t = -\sum_{k=1}^{V} y_{t,k} \log \hat{y}_{t,k}
$$

（$y_t$ 是下一词的 one-hot，$V$ 为词表大小，即交叉熵。）

上式属于 **many-to-many**：每个时间步都产生预测和损失。若做整句情感分类，通常使用最后有效状态 $h_T$，只计算一次 **many-to-one** 损失：

$$
\hat y=\mathrm{softmax}(W_ch_T+b_c), \qquad L=-\sum_k y_k\log\hat y_k
$$

## 4. 反向传播 BPTT 推导

### 4.1 先引入预激活量

为避免 tanh 的导数和隐藏状态梯度混在一起，先定义：

$$
a_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h, \qquad h_t=\tanh(a_t)
$$

再区分两个误差信号：

$$
\delta_t \triangleq \frac{\partial L}{\partial h_t}, \qquad g_t \triangleq \frac{\partial L}{\partial a_t}
$$

令 $D_t=\mathrm{diag}(1-h_t\odot h_t)$，则 $g_t=D_t\delta_t$。这里 $\delta_t$ 是对激活后状态的梯度，$g_t$ 是真正进入参数梯度的预激活误差。

### 4.2 输出路径与状态路径

softmax 与交叉熵给出：

$$
e_t \triangleq \hat{y}_t-y_t, \qquad \frac{\partial L_t}{\partial h_t}=W_{hy}^{\top}e_t
$$

而下一时刻的预激活量满足 $a_{t+1}=W_{xh}x_{t+1}+W_{hh}h_t+b_h$，所以未来误差传回 $h_t$ 时必须左乘 $W_{hh}^{\top}$：

$$
\delta_t = W_{hy}^{\top}e_t + W_{hh}^{\top}g_{t+1}
$$

### 4.3 BPTT 反向递推公式

结合 $g_t=D_t\delta_t$，得到更适合计算的递推式：

$$
\boxed{\;g_t=\left(W_{hy}^{\top}e_t+W_{hh}^{\top}g_{t+1}\right)\odot(1-h_t\odot h_t)\;}
$$

其中 $g_{T+1}=0$，从 $t=T$ 往回算到 $t=1$。这就是 Back-Propagation **Through Time**：除了当前输出路径，还多了一条跨时间返回的梯度路径。

### 4.4 参数梯度

因为参数是**共享**的（同一组 $W$ 参与了所有时刻），梯度要把每个时刻的贡献**求和**：

$$
\frac{\partial L}{\partial W_{hy}} = \sum_{t=1}^{T} e_t\, h_t^{\top}
$$

$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} g_t\, h_{t-1}^{\top}
$$

$$
\frac{\partial L}{\partial W_{xh}} = \sum_{t=1}^{T} g_t\, x_t^{\top}
$$

偏置项同样对时间求和：$\dfrac{\partial L}{\partial b_h}=\sum_t g_t$，$\dfrac{\partial L}{\partial b_y}=\sum_t e_t$。

> 记忆点：**前向共享权重 → 反向累积梯度**。

## 5. 梯度消失 / 爆炸的根源

沿时间传播的局部雅可比矩阵为：

$$
J_t \triangleq \frac{\partial h_t}{\partial h_{t-1}} = D_t W_{hh}, \qquad D_t=\mathrm{diag}(1-h_t\odot h_t)
$$

跨越多个时间步时，梯度会反复乘上 $J_t$：

$$
\frac{\partial h_t}{\partial h_k}=J_tJ_{t-1}\cdots J_{k+1}, \qquad k<t
$$

因此真正决定梯度大小的是一串**随时间变化的雅可比矩阵乘积**，既受 $W_{hh}$ 影响，也受 tanh 导数 $D_t$ 影响：

- 若这些乘积的范数持续小于 1，早期信息和梯度会快速衰减，形成**梯度消失**
- 若这些乘积的范数持续大于 1，梯度可能快速放大，形成**梯度爆炸**
- tanh 进入饱和区时 $1-h_t^2\approx0$，即使 $W_{hh}$ 较大也可能出现梯度消失

> 只有在线性 RNN 或固定工作点附近的线性化分析中，才能主要用 $W_{hh}$ 的谱半径讨论长期稳定性；对一般非线性 RNN，不能简单写成 $\rho(W_{hh})<1$ 必然消失、$\rho(W_{hh})>1$ 必然爆炸。

**与信号系统的对照**：RNN 是带反馈的非线性离散状态空间模型。去掉非线性或在工作点附近线性化后，它才与线性 IIR / 状态空间系统直接对应。BPTT 会沿展开后的状态链反向传播，序列越长，雅可比连乘越长，所以长期依赖更难学习。

## 6. 工程上的对策

| 问题 | 对策 |
|------|------|
| 梯度爆炸 | 梯度裁剪 `clip_grad_norm_` |
| 梯度消失 | LSTM / GRU 门控（用加法路径保护梯度） |
| 初始化 | 正交初始化、合理缩放 |
| 长序列 | 截断 BPTT（只回传最近 k 步） |

## 7. 与 PyTorch `nn.RNN` 对照

设 batch 大小为 $B$、序列长度为 $T$、词向量维度为 $n$、隐藏维度为 $d$、层数为 $L$，方向数为 $q$（单向 $q=1$，双向 $q=2$）：

```python
ids = torch.randint(0, vocab_size, (B, T))
x = embedding(ids)                         # (B, T, n)
output, h_n = rnn(x)                      # batch_first=True
# output: (B, T, q*d)  最后一层、所有时间步
# h_n:    (L*q, B, d)  每层每方向的最终状态
```

`output` 沿时间轴保留最后一层的 $[h_1,\dots,h_T]$；`h_n` 沿层数与方向保存最终状态。`batch_first=True` 只改变输入和 `output` 的维度顺序，不改变 `h_n`。单层单向且没有 PAD 干扰时，`output[:, -1, :]` 与 `h_n[-1]` 相同。

数学公式把偏置合写成一个 $b_h$；PyTorch 默认保存 `bias_ih_l0` 和 `bias_hh_l0` 两个偏置，它们在前向中相加，作用等价于一个总偏置。

### 7.1 多层 Dropout

`nn.RNN(..., num_layers=L, dropout=p)` 的内置 Dropout 只作用在相邻 RNN 层之间，不直接作用于同一层的 $h_{t-1}\to h_t$ 循环连接，也不作用在最后一层之后。因此 `num_layers=1` 时没有层间位置，内置 `dropout` 不生效。训练模式启用，`eval()` 模式关闭。

### 7.2 PAD 与最后有效状态

批量训练常把短句补到统一长度。此时 `output[:, -1, :]` 可能对应 PAD，而不是真实句子的最后一个词。应保存每条序列的真实长度，用 `pack_padded_sequence` 跳过 PAD，或按长度索引 `output[b, lengths[b]-1]`；计算逐时间步损失时也必须用 mask 排除 PAD 标签。

---

**下一单元**：用 PyTorch 手写前向递推和 BPTT，再与 `nn.RNN`、Autograd 的结果逐项核对。

In [1]:
# ============================================================
# 手写单层 RNN：核对前向结果，并用 Autograd 验证 BPTT 梯度
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

B, T, input_size, hidden_size, num_classes = 2, 3, 4, 3, 2
x = torch.randn(B, T, input_size, dtype=torch.float64)
targets = torch.tensor([[0, 1, 0], [1, 0, 1]])

# 数学记号中的 W_xh、W_hh、b_h、W_hy、b_y
W_xh = torch.randn(hidden_size, input_size, dtype=torch.float64, requires_grad=True)
W_hh = torch.randn(hidden_size, hidden_size, dtype=torch.float64, requires_grad=True)
b_h = torch.randn(hidden_size, dtype=torch.float64, requires_grad=True)
W_hy = torch.randn(num_classes, hidden_size, dtype=torch.float64, requires_grad=True)
b_y = torch.randn(num_classes, dtype=torch.float64, requires_grad=True)

# ---------- 1. 手写前向递推 ----------
h_prev = torch.zeros(B, hidden_size, dtype=torch.float64)
h_list, logit_list = [], []

for t in range(T):
    a_t = x[:, t] @ W_xh.T + h_prev @ W_hh.T + b_h
    h_t = torch.tanh(a_t)
    z_t = h_t @ W_hy.T + b_y
    h_list.append(h_t)
    logit_list.append(z_t)
    h_prev = h_t

states = torch.stack(h_list, dim=1)      # (B, T, hidden_size)
logits = torch.stack(logit_list, dim=1)  # (B, T, num_classes)
loss = F.cross_entropy(
    logits.reshape(-1, num_classes),
    targets.reshape(-1),
    reduction="sum",
)
loss.backward()

autograd_grads = {
    "W_xh": W_xh.grad.detach().clone(),
    "W_hh": W_hh.grad.detach().clone(),
    "b_h": b_h.grad.detach().clone(),
    "W_hy": W_hy.grad.detach().clone(),
    "b_y": b_y.grad.detach().clone(),
}

# ---------- 2. 按公式手写 BPTT ----------
with torch.no_grad():
    one_hot = F.one_hot(targets, num_classes=num_classes).to(torch.float64)
    errors = torch.softmax(logits, dim=-1) - one_hot  # e_t = y_hat_t - y_t

    dW_xh = torch.zeros_like(W_xh)
    dW_hh = torch.zeros_like(W_hh)
    db_h = torch.zeros_like(b_h)
    dW_hy = torch.zeros_like(W_hy)
    db_y = torch.zeros_like(b_y)
    g_next = torch.zeros(B, hidden_size, dtype=torch.float64)

    for t in reversed(range(T)):
        h_t = states[:, t]
        h_prev = states[:, t - 1] if t > 0 else torch.zeros_like(h_t)

        dW_hy += errors[:, t].T @ h_t
        db_y += errors[:, t].sum(dim=0)

        delta_h = errors[:, t] @ W_hy + g_next @ W_hh
        g_t = delta_h * (1.0 - h_t.square())

        dW_xh += g_t.T @ x[:, t]
        dW_hh += g_t.T @ h_prev
        db_h += g_t.sum(dim=0)
        g_next = g_t

manual_grads = {
    "W_xh": dW_xh, "W_hh": dW_hh, "b_h": db_h,
    "W_hy": dW_hy, "b_y": db_y,
}

# ---------- 3. 与 PyTorch nn.RNN 的前向结果核对 ----------
rnn = nn.RNN(input_size, hidden_size, batch_first=True).to(torch.float64)
with torch.no_grad():
    rnn.weight_ih_l0.copy_(W_xh)
    rnn.weight_hh_l0.copy_(W_hh)
    rnn.bias_ih_l0.copy_(b_h)
    rnn.bias_hh_l0.zero_()

rnn_output, h_n = rnn(x)

print(f"loss = {loss.item():.6f}")
print("手写前向 == nn.RNN output：", torch.allclose(states, rnn_output, atol=1e-12))
print("最后时间步 == h_n[-1]：", torch.allclose(states[:, -1], h_n[-1], atol=1e-12))
print("各参数的手写梯度与 Autograd 最大绝对误差：")
for name in manual_grads:
    max_error = (manual_grads[name] - autograd_grads[name]).abs().max().item()
    print(f"  {name:5s}: {max_error:.3e}")

loss = 8.807888
手写前向 == nn.RNN output： True
最后时间步 == h_n[-1]： True
各参数的手写梯度与 Autograd 最大绝对误差：
  W_xh : 3.053e-16
  W_hh : 6.245e-17
  b_h  : 1.110e-16
  W_hy : 2.220e-16
  b_y  : 2.220e-16
